# Source Fingerprinter (Pollution Source Apportionment)
Uses XGBoost to infer the likely source of pollution based on chemical profile and temporal data.

## 1. Dataset Preparation
We generate mock OpenAQ data, add derived PM variables, and generate target labels based on heuristic thresholds for Vehicular, Industrial, Construction, Biomass, and Mixed signatures.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed and create temporal index
np.random.seed(42)
dates = pd.date_range(start='2022-01-01', end='2024-03-01', freq='h')
n_samples = len(dates)

# Generate mock OpenAQ feature space
df = pd.DataFrame({
    'timestamp': dates,
    'pm25': np.random.uniform(10, 200, n_samples),
    'pm10': np.random.uniform(20, 300, n_samples),
    'no2': np.random.uniform(10, 80, n_samples),
    'so2': np.random.uniform(5, 50, n_samples),
    'hour': dates.hour,
    'month': dates.month,
    'is_weekend': (dates.dayofweek >= 5).astype(int),
})

# Generate missing requested features
df['pm1'] = df['pm25'] * np.random.uniform(0.4, 0.8, n_samples)
df['pm1_pm25_ratio'] = df['pm1'] / df['pm25']
df['pm10_pm25_ratio'] = df['pm10'] / df['pm25']
df['wind_dir_sector'] = np.random.randint(0, 8, n_samples)

def classify_source(row):
    h = row['hour']
    m = row['month']
    pm25 = row['pm25']
    pm10 = row['pm10']
    no2 = row['no2']
    so2 = row['so2']
    
    if no2 > 60 and h in [7, 8, 9, 17, 18, 19, 20] and (pm10 / pm25) < 2.5:
        return 0 # Vehicular
    if so2 > 30 and h in [6, 7, 8, 20, 21, 22]:
        return 1 # Industrial
    if pm10 > 150 and (pm10 / pm25) > 3.0 and (8 <= h <= 18):
        return 2 # Construction
    if pm25 > 100 and so2 < 20 and no2 < 40 and m in [10, 11]:
        return 3 # Biomass
    return 4 # Mixed/Unknown

df['source_label'] = df.apply(classify_source, axis=1)

print("Class Distribution:")
dist = df['source_label'].value_counts().sort_index()
print(dist)

if dist.min() / dist.max() < 0.1:
    print("\n[FLAG] Imbalance Detected: Some classes contain < 10% of majority class.")

## 2. Handling Imbalance & Feature Preprocessing
Select final schema and utilize `imblearn.over_sampling.SMOTE` to synthesize data for minority classes.

In [ ]:
!pip install -q imbalanced-learn xgboost shap

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

features = [
    'pm25', 'pm10', 'pm1', 'no2', 'so2', 
    'pm1_pm25_ratio', 'pm10_pm25_ratio', 
    'wind_dir_sector', 'hour', 'month', 'is_weekend'
]

X = df[features]
y = df['source_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Original training shape: {X_train.shape}")
print(y_train.value_counts().sort_index())

# SMOTE initialization
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nResampled training shape: {X_train_resampled.shape}")
print(y_train_resampled.value_counts().sort_index())

## 3. XGBoost Model Training
Initializes an `XGBClassifier` and executes GridSearchCV on a 3x3 dimension configuration (max_depth and learning_rate).

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(
    n_estimators=300, 
    max_depth=6, 
    learning_rate=0.05, 
    subsample=0.8,
    random_state=42,
    verbosity=0
)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_resampled, y_train_resampled)

best_model = grid_search.best_estimator_
print(f"\nBest Parameters Found: {grid_search.best_params_}")

## 4. Evaluation
Analyze precision, recall, F1 via classification report, plot a raw confusion matrix, and explain feature impacts using SHAP.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import shap
import warnings
warnings.filterwarnings('ignore')

# Perform held-out predictions
y_pred = best_model.predict(X_test)
label_names = ['Vehicular', 'Industrial', 'Construction', 'Biomass', 'Mixed/Unknown']

print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_names))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix on Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# SHAP Extractor
print("Computing SHAP values... (this may take a moment)")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.title("Top 10 Important Features (SHAP Summary)")
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10)
plt.show()

## 5. Artifact Saving
Outputs native XGB JSON along with our encoder maps so backend services can easily translate inputs & outputs.

In [ ]:
import os
import json

os.makedirs("models_saved", exist_ok=True)

# Save XGB object intrinsically
best_model.save_model("models_saved/source_fingerprinter.json")

metadata = {
    "features": features,
    "labels": {
        0: "Vehicular",
        1: "Industrial",
        2: "Construction",
        3: "Biomass",
        4: "Mixed/Unknown"
    }
}

# Save Schema map
with open("models_saved/fingerprinter_meta.json", "w") as f:
    json.dump(metadata, f, indent=4)
    
print("Artifacts exported to 'models_saved/' directory.")